In [1]:
using Revise

using LinearAlgebra
using SparseArrays
using Dates
using Random
using ProgressBars

includet("../julia/src/UnitaryDilation/UnitaryDilation.jl")
includet("../julia/src/PetzMaps.jl")
includet("../julia/src/utils.jl")

using .UnitaryDilation
using .PetzMaps

## Utility Functions

In [2]:
function get_kraus_operators(noise, gamma, t)
  if noise == "amplitude_damping"
    return get_amplitudedamping_operators(gamma, t)
  elseif noise == "dephasing"
    return get_dephasing_operators(gamma, t)
  elseif noise == "bitflip"
    return get_bitflip_operators(gamma, t)
  else
    error("Unknown noise model: $noise")
  end

end

function apply_noise(model, ρ, n_qubits)
  ρf = apply_channel(model.kraus_fwd, ρ, n_qubits)
  # Enforce physicality (hermitianicity and trace 1)
  enforce_physical!(ρf)
  return ρf
end


function recovery(model, ρ)
  ρr, η = apply_petz_collision(model, ρ)
  enforce_physical!(ρr)
  return ρr, η
end

recovery (generic function with 1 method)

## Prove Autorecovery
Select a starting (thermal/random) state and prove if the Petz map can recover it after the application of noise

In [3]:
n_qubits = 1
beta = 0.5
gamma = 1.0
dt = 0.1
noise = "amplitude_damping"
n_steps = 2

2

Initialize the system

In [4]:
# Choose a reference state for the recovery
sigma = thermal_state(n_qubits, beta)
# The initial state is the reference state itself
ρ0 = sigma
sigma

2×2 Matrix{ComplexF64}:
 0.5+0.0im  0.0+0.0im
 0.0-0.0im  0.5+0.0im

In [5]:
# Take the relevant Kraus operators
kraus_single_qubit = get_kraus_operators(noise, gamma, dt)
# Build the Petz collision model
# this object store the Kraus operators and the collision Unitary
collision_model = PetzCollisionModel(kraus_single_qubit, sigma, n=n_qubits)

PetzCollisionModel{ComplexF64}(2, 2, ComplexF64[0.5 + 0.0im 0.0 + 0.0im; 0.0 - 0.0im 0.5 + 0.0im], Matrix{ComplexF64}[[1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.951229424500714 + 0.0im], [0.0 + 0.0im 0.3084843301758462 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im]], Matrix{ComplexF64}[[0.95556602804532 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.999999999889483 + 0.0im], [0.0 + 0.0im 0.0 + 0.0im; 0.2947771461003544 + 0.0im 0.0 + 0.0im]], ComplexF64[-0.9555660281325733 + 0.0im 0.0 + 0.0im 0.0 + 0.0im -0.29477714612727063 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im -1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im -1.0 + 0.0im 0.0 + 0.0im 0.0 + 0.0im; -0.29477714612727063 + 0.0im 0.0 + 0.0im 0.0 + 0.0im 0.9555660281325732 + 0.0im])

### Step 1
We apply the noise and recover the state, both with the collision model and the standard Petz map.
- `rho1`: control state where only noise is applied,
- `rho2`: state where noise + collision is applied,
- `rho3`: state where noise + Petz map is applied
---
First, we check that the Kraus operators extension performed in the `PetzCollisionModel` constructor is valid.
While our standard is to apply single-qubit Kraus operators to each qubit independently, this extension creates global Kraus operators for the multipartite state.

In [6]:
rho1 = copy(ρ0)
rho2 = copy(ρ0)
rho3 = copy(ρ0)

rho1 = apply_channel(kraus_single_qubit, rho1, n_qubits)
rho2 = apply_channel(kraus_single_qubit, rho2, n_qubits)
rho3 = apply_channel(collision_model.kraus_fwd, rho3)

if isapprox(rho1, rho2) && isapprox(rho2, rho3)
    println("All methods give the same result.")
else
    println("Results differ between methods.")
end

All methods give the same result.


Then, we recover `rho2` and `rho3` with the two different methods.

In [ ]:
rho2 = recovery_map(kraus_single_qubit, sigma, rho2, n_qubits)
rho3, η = apply_petz_collision(collision_model, rho3)

f1 = fidelity(rho1, sigma)
f2 = fidelity(rho2, sigma)
f3 = fidelity(rho3, sigma)
println("Fidelity after recovery map: ", f2)
println("Fidelity after Petz collision: ", f3)

Fidelity after recovery map: 1.0000000000000004
Fidelity after Petz collision: 1.0000000000000004


Now we can iterate in time

In [ ]:
steps = 10
f1s = Float64[f1]
f2s = Float64[f2]
f3s = Float64[f3]
for i in 2:steps
    println("Step $i")
    rho1 = apply_channel(kraus_single_qubit, rho1, n_qubits)
    
    rho2 = apply_channel(kraus_single_qubit, rho2, n_qubits)
    rho2 = recovery_map(kraus_single_qubit, sigma, rho2, n_qubits)
    
    rho3 = apply_channel(collision_model.kraus_fwd, rho3, n_qubits)
    rho3, η = apply_petz_collision(collision_model, rho3)
    
    push!(f1s, fidelity(rho1, sigma))
    push!(f2s, fidelity(rho2, sigma))
    push!(f3s, fidelity(rho3, sigma))
end
println("Fidelity after direct noise: ", fidelity(rho1, sigma))
println("Fidelity after recovery map: ", fidelity(rho2, sigma))
println("Fidelity after Petz collision: ", fidelity(rho3, sigma))

Step 2
Fidelity after direct noise: 0.98291427947137
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 3
Fidelity after direct noise: 0.9720463769467088
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 4
Fidelity after direct noise: 0.9596688694739468
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 5
Fidelity after direct noise: 0.9462143712039797
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 6
Fidelity after direct noise: 0.9320224657472147
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 7
Fidelity after direct noise: 0.9173611775907614
Fidelity after recovery map: 1.0000000000000009
Fidelity after Petz collision: 1.0000000000000009
Step 8
Fidelity after direct noise: 0.9024426764334309
Fidelity after recovery map: 